# Full-Haar fidelity-informed NMR pulse compiler — nominal model

This notebook trains a **single neural pulse compiler** for the full single-qubit gate family using fresh Haar-random \(SU(2)\) targets.

It uses the nominal three-spin Hamiltonian

\[
H_0=\pi\sum_i v_i Z_i+
\pi\sum_{i<j}J_{ij}(X_iX_j+Y_iY_j+Z_iZ_j),
\]

with the manuscript parameters

- \(v=(-921,\ 40.75,\ 700)\) Hz
- \(J_{12}=-64\) Hz, \(J_{13}=24.4\) Hz, \(J_{23}=34.1\) Hz
- \(T=300\) slices
- manuscript value \(\Delta t=35\) ns

The neural network outputs **two RF quadratures per slice** \((u_x,u_y)\). Thus amplitude and phase are both learned:

\[
A_t=\sqrt{u_{x,t}^2+u_{y,t}^2},\qquad
\phi_t=\operatorname{atan2}(u_{y,t},u_{x,t}).
\]

**Important experimental point:** the manuscript does not provide the calibrated maximum RF amplitude. `rabi_max_hz` below is therefore an explicit provisional numerical cap, not an extracted experimental quantity. Replace it with the measured hardware limit before using the resulting amplitudes experimentally.

Also, the notebook deliberately keeps the manuscript's `35 ns` slice duration but prints a physical-selectivity warning. We should verify that unit against the experimental pulse files before the rewritten paper is finalized.

In [ ]:
# Colab GPU check
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Select Runtime -> Change runtime type -> GPU in Colab before training.")


In [ ]:
"""
Full-Haar fidelity-informed neural pulse compiler for a 3-spin NMR processor.
Nominal (no robustness) version.

Physics follows the uploaded manuscript's nominal Hamiltonian:
H0 = pi sum_i v_i Z_i + pi sum_{i<j} J_ij (X_i X_j + Y_i Y_j + Z_i Z_j)
with one global transverse RF channel.

The network receives a Haar-random SU(2) quaternion q=(q0,qx,qy,qz)
and outputs both RF quadratures (u_x,u_y) at every time slice.  Amplitude and
phase are recovered as A=sqrt(u_x^2+u_y^2), phi=atan2(u_y,u_x).
"""

from __future__ import annotations

import math
import os
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


@dataclass
class Config:
    # ---------- manuscript experimental parameters ----------
    v_hz: Tuple[float, float, float] = (-921.0, 40.75, 700.0)
    j_hz: Tuple[float, float, float] = (-64.0, 24.4, 34.1)  # J12,J13,J23
    n_slices: int = 300
    dt_s: float = 35e-9  # manuscript explicitly says 35 ns

    # The manuscript does NOT report the calibrated RF amplitude ceiling.
    # This is therefore a provisional numerical cap and MUST be replaced by
    # the measured hardware value before making an experimental-amplitude claim.
    rabi_max_hz: float = 50_000.0

    # ---------- neural compiler ----------
    width: int = 256
    hidden_layers: int = 5
    dropout: float = 0.10

    # ---------- training ----------
    batch_size: int = 32
    train_steps: int = 5000
    lr: float = 5e-4
    weight_decay: float = 1e-3
    grad_clip: float = 1.0

    # Fidelity is the main objective. These only mildly regularize waveform roughness/power.
    smooth_lambda: float = 1e-4
    power_lambda: float = 1e-6

    val_gates: int = 512
    val_batch: int = 64
    validate_every: int = 100
    print_every: int = 20

    seed: int = 20260917
    out_dir: str = "full_haar_nominal_run"


CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def kron3(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    return torch.kron(torch.kron(a, b), c)


class NMRSystem:
    def __init__(self, cfg: Config, device: torch.device):
        self.cfg = cfg
        self.device = device
        ctype = torch.complex64

        I = torch.eye(2, dtype=ctype, device=device)
        X = torch.tensor([[0, 1], [1, 0]], dtype=ctype, device=device)
        Y = torch.tensor([[0, -1j], [1j, 0]], dtype=ctype, device=device)
        Z = torch.tensor([[1, 0], [0, -1]], dtype=ctype, device=device)

        self.I2 = I
        self.I8 = torch.eye(8, dtype=ctype, device=device)
        self.I4 = torch.eye(4, dtype=ctype, device=device)
        self.X, self.Y, self.Z = X, Y, Z

        self.Xs = [kron3(X, I, I), kron3(I, X, I), kron3(I, I, X)]
        self.Ys = [kron3(Y, I, I), kron3(I, Y, I), kron3(I, I, Y)]
        self.Zs = [kron3(Z, I, I), kron3(I, Z, I), kron3(I, I, Z)]

        self.Hx = self.Xs[0] + self.Xs[1] + self.Xs[2]
        self.Hy = self.Ys[0] + self.Ys[1] + self.Ys[2]

        v1, v2, v3 = cfg.v_hz
        j12, j13, j23 = cfg.j_hz
        H0 = math.pi * (v1 * self.Zs[0] + v2 * self.Zs[1] + v3 * self.Zs[2])
        for (i, j, Jij) in [(0, 1, j12), (0, 2, j13), (1, 2, j23)]:
            H0 = H0 + math.pi * Jij * (
                self.Xs[i] @ self.Xs[j]
                + self.Ys[i] @ self.Ys[j]
                + self.Zs[i] @ self.Zs[j]
            )
        self.H0 = H0

    def diagnostics(self) -> Dict[str, float]:
        cfg = self.cfg
        total_t = cfg.n_slices * cfg.dt_s
        max_offset_sep = max(cfg.v_hz) - min(cfg.v_hz)
        drift_phase = math.pi * max(abs(x) for x in (*cfg.v_hz, *cfg.j_hz)) * total_t
        selectivity_phase = math.pi * abs(max_offset_sep) * total_t
        herm_err = torch.linalg.norm(self.H0 - self.H0.mH).item()
        return {
            "total_pulse_s": total_t,
            "max_offset_separation_hz": max_offset_sep,
            "largest_drift_action_rad": drift_phase,
            "offset_discrimination_action_rad": selectivity_phase,
            "H0_hermiticity_error": herm_err,
        }


@torch.no_grad()
def sample_haar_quaternions(n: int, device: torch.device) -> torch.Tensor:
    """Haar-uniform SU(2): a normalized N(0,1)^4 vector is uniform on S^3."""
    q = torch.randn(n, 4, device=device, dtype=torch.float32)
    return q / q.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def quaternion_to_u2(q: torch.Tensor, sys: NMRSystem) -> torch.Tensor:
    """U = q0 I - i(qx X + qy Y + qz Z)."""
    q0, qx, qy, qz = q.unbind(dim=-1)
    I, X, Y, Z = sys.I2, sys.X, sys.Y, sys.Z
    U = (
        q0[:, None, None].to(I.dtype) * I
        - 1j * (
            qx[:, None, None].to(I.dtype) * X
            + qy[:, None, None].to(I.dtype) * Y
            + qz[:, None, None].to(I.dtype) * Z
        )
    )
    return U


def lift_target(U2: torch.Tensor, sys: NMRSystem) -> torch.Tensor:
    # kron(U2, I4), batched: indices (i,k),(j,l)
    return torch.einsum("bij,kl->bikjl", U2, sys.I4).reshape(-1, 8, 8)


class PulseCompiler(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        layers = []
        in_dim = 4
        for _ in range(cfg.hidden_layers):
            layers += [nn.Linear(in_dim, cfg.width), nn.GELU()]
            if cfg.dropout > 0:
                layers.append(nn.Dropout(cfg.dropout))
            in_dim = cfg.width
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, 2 * cfg.n_slices)
        self.cfg = cfg

        # Keep the initial waveform moderate rather than saturating the RF cap.
        nn.init.normal_(self.head.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.head.bias)

    def forward(self, q: torch.Tensor) -> torch.Tensor:
        raw = self.head(self.backbone(q)).view(-1, self.cfg.n_slices, 2)
        # Smooth radial map R^2 -> open disk of radius rabi_max_hz.
        # ||u|| = Amax * r/sqrt(1+r^2) < Amax.
        r2 = (raw * raw).sum(dim=-1, keepdim=True)
        controls = self.cfg.rabi_max_hz * raw / torch.sqrt(1.0 + r2)
        return controls


def propagate(controls_hz: torch.Tensor, sys: NMRSystem) -> torch.Tensor:
    """
    controls_hz[...,0:2] = equivalent transverse Rabi coefficients in Hz.
    We use Hc = pi * (u_x Hx + u_y Hy), so the manuscript's A(t) is
    A_rad/s = pi * sqrt(u_x^2 + u_y^2).
    """
    B, T, two = controls_hz.shape
    assert T == sys.cfg.n_slices and two == 2
    ux = controls_hz[..., 0]
    uy = controls_hz[..., 1]

    H = (
        sys.H0[None, None, :, :]
        + math.pi * ux[..., None, None].to(sys.H0.dtype) * sys.Hx[None, None, :, :]
        + math.pi * uy[..., None, None].to(sys.H0.dtype) * sys.Hy[None, None, :, :]
    )
    U_steps = torch.matrix_exp((-1j * sys.cfg.dt_s) * H)

    U = sys.I8.unsqueeze(0).expand(B, -1, -1).clone()
    for t in range(T):
        U = U_steps[:, t] @ U
    return U


def unitary_fidelity(target: torch.Tensor, actual: torch.Tensor) -> torch.Tensor:
    # Tr(target^dagger actual) = Frobenius inner product <target, actual>
    overlap = (target.conj() * actual).sum(dim=(-2, -1))
    return overlap.abs().square() / 64.0


def loss_terms(controls_hz: torch.Tensor, fids: torch.Tensor, cfg: Config):
    fid_loss = 1.0 - fids.mean()
    u = controls_hz / cfg.rabi_max_hz
    smooth = (u[:, 1:] - u[:, :-1]).square().sum(dim=-1).mean()
    power = u.square().sum(dim=-1).mean()
    total = fid_loss + cfg.smooth_lambda * smooth + cfg.power_lambda * power
    return total, fid_loss, smooth, power


@torch.no_grad()
def evaluate(model: nn.Module, sys: NMRSystem, q_val: torch.Tensor, batch_size: int) -> np.ndarray:
    model.eval()
    out = []
    for s in range(0, len(q_val), batch_size):
        q = q_val[s:s+batch_size]
        target = lift_target(quaternion_to_u2(q, sys), sys)
        controls = model(q)
        U = propagate(controls, sys)
        out.append(unitary_fidelity(target, U).detach().cpu())
    return torch.cat(out).numpy()


def summarize_fids(fids: np.ndarray) -> Dict[str, float]:
    return {
        "mean": float(np.mean(fids)),
        "median": float(np.median(fids)),
        "p05": float(np.percentile(fids, 5)),
        "p01": float(np.percentile(fids, 1)),
        "min": float(np.min(fids)),
        "max": float(np.max(fids)),
    }


def train(cfg: Config = CFG):
    set_seed(cfg.seed)
    out_dir = Path(cfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not torch.cuda.is_available():
        print("WARNING: CUDA GPU not found. This code is intended for Colab GPU runtime.")
        device = torch.device("cpu")
    else:
        device = torch.device("cuda")
        torch.backends.cuda.matmul.allow_tf32 = True
        print("GPU:", torch.cuda.get_device_name(0))

    sys = NMRSystem(cfg, device)
    print("\nSYSTEM DIAGNOSTICS")
    for k, v in sys.diagnostics().items():
        print(f"  {k}: {v}")
    if sys.diagnostics()["offset_discrimination_action_rad"] < 0.2:
        print(
            "\nIMPORTANT WARNING: with the manuscript's dt=35 ns and T=300, "
            "the chemical-shift discrimination accumulated over the whole pulse is very small. "
            "If this run cannot learn selective U1⊗I⊗I control, verify experimentally whether "
            "the intended slice duration is 35 ns or another unit before changing the model.\n"
        )

    # Exact full interaction check
    assert torch.allclose(sys.H0, sys.H0.mH, atol=1e-5), "H0 is not Hermitian"

    model = PulseCompiler(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.train_steps, eta_min=cfg.lr * 0.05)

    q_val = sample_haar_quaternions(cfg.val_gates, device)
    best_mean = -1.0
    history = []

    for step in range(1, cfg.train_steps + 1):
        model.train()
        q = sample_haar_quaternions(cfg.batch_size, device)
        target = lift_target(quaternion_to_u2(q, sys), sys)
        controls = model(q)
        U = propagate(controls, sys)
        fids = unitary_fidelity(target, U)
        loss, fid_loss, smooth, power = loss_terms(controls, fids, cfg)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        if step % cfg.print_every == 0 or step == 1:
            print(
                f"step {step:5d}/{cfg.train_steps} | loss={loss.item():.6f} "
                f"| train F={fids.mean().item():.6f} | min={fids.min().item():.6f} "
                f"| lr={scheduler.get_last_lr()[0]:.2e}"
            )

        if step % cfg.validate_every == 0 or step == cfg.train_steps:
            vf = evaluate(model, sys, q_val, cfg.val_batch)
            stats = summarize_fids(vf)
            row = {"step": step, **stats}
            history.append(row)
            print("  VAL:", ", ".join(f"{k}={v:.6f}" for k, v in stats.items()))

            ckpt = {
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "config": asdict(cfg),
                "val_stats": stats,
            }
            torch.save(ckpt, out_dir / "last.pt")
            if stats["mean"] > best_mean:
                best_mean = stats["mean"]
                torch.save(ckpt, out_dir / "best.pt")
                print("  saved new best.pt")

    # Final evaluation using the best checkpoint.
    best = torch.load(out_dir / "best.pt", map_location=device)
    model.load_state_dict(best["model"])
    final_fids = evaluate(model, sys, q_val, cfg.val_batch)
    final_stats = summarize_fids(final_fids)
    print("\nFINAL BEST VALIDATION")
    print(final_stats)

    np.save(out_dir / "validation_fidelities.npy", final_fids)
    np.save(out_dir / "validation_quaternions.npy", q_val.cpu().numpy())
    np.save(out_dir / "history.npy", np.array(history, dtype=object), allow_pickle=True)

    # Save one representative pulse as both I/Q and amplitude/phase.
    model.eval()
    q_demo = q_val[:1]
    controls = model(q_demo)[0].detach().cpu().numpy()
    ux, uy = controls[:, 0], controls[:, 1]
    amp_hz = np.sqrt(ux**2 + uy**2)
    phase_rad = np.arctan2(uy, ux)
    np.savez(
        out_dir / "demo_pulse.npz",
        quaternion=q_demo[0].cpu().numpy(),
        ux_hz=ux,
        uy_hz=uy,
        amplitude_hz=amp_hz,
        phase_rad=phase_rad,
        dt_s=cfg.dt_s,
    )

    # Plots
    plt.figure(figsize=(6, 4))
    plt.hist(final_fids, bins=40)
    plt.xlabel("Unitary fidelity")
    plt.ylabel("Count")
    plt.title("Held-out Haar SU(2) validation")
    plt.tight_layout()
    plt.savefig(out_dir / "fidelity_hist.png", dpi=180)
    plt.show()

    t_us = np.arange(cfg.n_slices) * cfg.dt_s * 1e6
    plt.figure(figsize=(8, 4))
    plt.plot(t_us, amp_hz)
    plt.xlabel("Time (microseconds)")
    plt.ylabel("RF amplitude (Hz-equivalent)")
    plt.title("Compiled pulse amplitude")
    plt.tight_layout()
    plt.savefig(out_dir / "demo_amplitude.png", dpi=180)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(t_us, np.unwrap(phase_rad))
    plt.xlabel("Time (microseconds)")
    plt.ylabel("Unwrapped phase (rad)")
    plt.title("Compiled pulse phase")
    plt.tight_layout()
    plt.savefig(out_dir / "demo_phase.png", dpi=180)
    plt.show()

    return model, sys, final_stats




## Main configuration

This keeps the experimental values in the manuscript. The only provisional hardware number is `rabi_max_hz`, because the paper does not report an RF ceiling.

In [ ]:
CFG = Config(
    # Experimental parameters from the manuscript
    v_hz=(-921.0, 40.75, 700.0),
    j_hz=(-64.0, 24.4, 34.1),
    n_slices=300,
    dt_s=35e-6,              # exact value written in the current manuscript

    # NOT reported in the manuscript: replace with calibrated hardware limit.
    rabi_max_hz=50_000.0,

    # Network: similar MLP family to the paper, now outputting 2*T controls
    width=256,
    hidden_layers=10,
    dropout=0.25,

    # Training
    batch_size=32,
    train_steps=50000,
    lr=5e-4,
    weight_decay=1e-3,
    grad_clip=1.0,

    # Fidelity remains the dominant loss
    smooth_lambda=1e-4,
    power_lambda=1e-6,

    # Held-out Haar validation
    val_gates=512,
    val_batch=64,
    validate_every=100,
    print_every=20,

    seed=20260917,
    out_dir="outputs/nominal",
)

print(CFG)


## Physics/self-check before training

This checks Hermiticity, target-gate unitarity, propagator unitarity, tensor shapes, and one complete differentiable backward pass.

In [ ]:
set_seed(CFG.seed)
device = torch.device("cuda")
sys_nmr = NMRSystem(CFG, device)
model_test = PulseCompiler(CFG).to(device)

print("System diagnostics:")
for k, v in sys_nmr.diagnostics().items():
    print(f"  {k}: {v}")

q = sample_haar_quaternions(2, device)
U2 = quaternion_to_u2(q, sys_nmr)
target = lift_target(U2, sys_nmr)
controls = model_test(q)
U = propagate(controls, sys_nmr)
fids = unitary_fidelity(target, U)
loss, *_ = loss_terms(controls, fids, CFG)
loss.backward()

target_unitarity = torch.linalg.norm(target.mH @ target - sys_nmr.I8).item()
actual_unitarity = torch.linalg.norm(U.mH @ U - sys_nmr.I8).item()

print("\nShapes:")
print("  q:", q.shape)
print("  target:", target.shape)
print("  controls:", controls.shape)
print("  U:", U.shape)
print("\nChecks:")
print("  H0 Hermiticity error:", torch.linalg.norm(sys_nmr.H0 - sys_nmr.H0.mH).item())
print("  target unitarity error:", target_unitarity)
print("  propagated unitarity error:", actual_unitarity)
print("  demo fidelities:", fids.detach().cpu().numpy())
print("  gradient norm:", model_test.head.weight.grad.norm().item())

assert target_unitarity < 1e-4
assert actual_unitarity < 1e-3
assert torch.isfinite(model_test.head.weight.grad).all()
print("\nSELF CHECK PASSED")
del model_test, q, U2, target, controls, U
torch.cuda.empty_cache()


## Train

Every training batch contains newly sampled Haar-random gates, so this is not interpolation over a fixed angle grid or a finite labeled pulse set. Supervision comes only from the differentiable physical unitary fidelity.

In [ ]:
model, sys_nmr, final_stats = train(CFG)


## Load the best model later

In [ ]:
# Example reload
ckpt = torch.load(f"{CFG.out_dir}/best.pt", map_location="cuda")
model = PulseCompiler(CFG).cuda()
model.load_state_dict(ckpt["model"])
model.eval()
print("Best validation statistics:", ckpt["val_stats"])


## Compile any new Haar gate and export amplitude/phase

In [ ]:
with torch.no_grad():
    q_new = sample_haar_quaternions(1, torch.device("cuda"))
    ctrl = model(q_new)[0].cpu().numpy()

ux = ctrl[:, 0]
uy = ctrl[:, 1]
amplitude_hz = np.sqrt(ux**2 + uy**2)
phase_rad = np.arctan2(uy, ux)

print("Quaternion:", q_new[0].cpu().numpy())
print("Peak amplitude (Hz-equivalent):", amplitude_hz.max())
print("Amplitude shape:", amplitude_hz.shape)
print("Phase shape:", phase_rad.shape)

np.savez(
    "/content/compiled_pulse.npz",
    quaternion=q_new[0].cpu().numpy(),
    ux_hz=ux,
    uy_hz=uy,
    amplitude_hz=amplitude_hz,
    phase_rad=phase_rad,
    dt_s=CFG.dt_s,
)
print("Saved /content/compiled_pulse.npz")


In [ ]:
# ============================================================
# COMPLETE PAPER / REPRODUCIBILITY EXPORT
# Run this cell AFTER:
#     model, sys_nmr, final_stats = train(CFG)
#
# It creates one ZIP containing:
#   - best + last checkpoints
#   - model weights
#   - optimizer state contained in checkpoints
#   - exact configuration
#   - full training history (NPY + CSV)
#   - validation fidelities + Haar quaternions
#   - exact H0 Hamiltonian
#   - model architecture / parameter count
#   - GPU / Python / PyTorch / pip environment
#   - representative worst / median / best pulses
#   - target + achieved unitaries for those pulses
#   - paper-ready PNG + vector PDF plots
#   - summary + manifest
#   - final ZIP download
# ============================================================

import os
import json
import math
import shutil
import platform
import subprocess
import sys as pysys
from pathlib import Path
from datetime import datetime
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt



# ------------------------------------------------------------
# 0. Locate the completed run
# ------------------------------------------------------------

RUN_DIR = Path(CFG.out_dir)

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory does not exist: {RUN_DIR}")

BEST_CKPT = RUN_DIR / "best.pt"
LAST_CKPT = RUN_DIR / "last.pt"
HISTORY_FILE = RUN_DIR / "history.npy"
FID_FILE = RUN_DIR / "validation_fidelities.npy"
Q_FILE = RUN_DIR / "validation_quaternions.npy"

required = [BEST_CKPT, HISTORY_FILE, FID_FILE, Q_FILE]

for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"Required training artifact missing: {p}\n"
            "Make sure the training run finished normally."
        )


# ------------------------------------------------------------
# 1. Create clean export directory
# ------------------------------------------------------------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

EXPORT_ROOT = Path(f"/content/full_haar_nominal_paper_export_{stamp}")
RAW_DIR = EXPORT_ROOT / "raw_training_artifacts"
FIG_DIR = EXPORT_ROOT / "figures"
TABLE_DIR = EXPORT_ROOT / "tables"
PULSE_DIR = EXPORT_ROOT / "representative_pulses"
MODEL_DIR = EXPORT_ROOT / "model"
ENV_DIR = EXPORT_ROOT / "environment"

for d in [
    EXPORT_ROOT,
    RAW_DIR,
    FIG_DIR,
    TABLE_DIR,
    PULSE_DIR,
    MODEL_DIR,
    ENV_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Export directory:")
print(EXPORT_ROOT)


# ------------------------------------------------------------
# 2. Copy ALL original run artifacts
# ------------------------------------------------------------

for item in RUN_DIR.iterdir():
    if item.is_file():
        shutil.copy2(item, RAW_DIR / item.name)

print(f"\nCopied raw artifacts from {RUN_DIR}")


# ------------------------------------------------------------
# 3. Load best checkpoint and exact validation data
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_ckpt = torch.load(
    BEST_CKPT,
    map_location=device,
    weights_only=False
)

# Guarantee that the model corresponds to BEST validation checkpoint.
model.load_state_dict(best_ckpt["model"])
model = model.to(device)
model.eval()

fids = np.load(FID_FILE).astype(np.float64)
q_val_np = np.load(Q_FILE).astype(np.float64)

assert len(fids) == len(q_val_np)

history_raw = np.load(
    HISTORY_FILE,
    allow_pickle=True
)

history = [
    dict(x) if not isinstance(x, dict) else x
    for x in history_raw
]

history_df = pd.DataFrame(history)


# ------------------------------------------------------------
# 4. Recompute authoritative summary from stored fidelities
# ------------------------------------------------------------

summary = {
    "n_validation_gates": int(len(fids)),
    "mean": float(np.mean(fids)),
    "std": float(np.std(fids)),
    "median": float(np.median(fids)),
    "p05": float(np.quantile(fids, 0.05)),
    "p01": float(np.quantile(fids, 0.01)),
    "min": float(np.min(fids)),
    "max": float(np.max(fids)),
    "mean_infidelity": float(np.mean(1.0 - fids)),
    "median_infidelity": float(np.median(1.0 - fids)),
}

# Useful paper thresholds.
for threshold in [0.90, 0.95, 0.97, 0.98, 0.99, 0.995]:
    summary[f"fraction_F_ge_{threshold:.3f}"] = float(
        np.mean(fids >= threshold)
    )

with open(EXPORT_ROOT / "final_validation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)


# ------------------------------------------------------------
# 5. Save exact configuration
# ------------------------------------------------------------

if "config" in best_ckpt:
    config_dict = best_ckpt["config"]
elif is_dataclass(CFG):
    config_dict = asdict(CFG)
else:
    config_dict = vars(CFG)

with open(EXPORT_ROOT / "config.json", "w") as f:
    json.dump(config_dict, f, indent=2, default=str)


# ------------------------------------------------------------
# 6. Save model in several useful forms
# ------------------------------------------------------------

torch.save(
    model.state_dict(),
    MODEL_DIR / "best_model_state_dict.pt"
)

# Complete checkpoint already contains:
# model + optimizer + step + config + validation stats.
shutil.copy2(
    BEST_CKPT,
    MODEL_DIR / "best_full_checkpoint.pt"
)

if LAST_CKPT.exists():
    shutil.copy2(
        LAST_CKPT,
        MODEL_DIR / "last_full_checkpoint.pt"
    )

# Human-readable architecture.
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

with open(MODEL_DIR / "architecture.txt", "w") as f:
    f.write(str(model))
    f.write("\n\n")
    f.write(f"Total parameters: {n_params:,}\n")
    f.write(f"Trainable parameters: {n_trainable:,}\n")

# Try to save a portable traced inference model.
try:
    example_q = torch.tensor(
        q_val_np[:1],
        dtype=torch.float32,
        device=device
    )

    traced_model = torch.jit.trace(
        model,
        example_q
    )

    traced_model.save(
        str(MODEL_DIR / "pulse_compiler_traced.pt")
    )

    traced_saved = True

except Exception as exc:
    traced_saved = False
    with open(MODEL_DIR / "tracing_error.txt", "w") as f:
        f.write(str(exc))


# ------------------------------------------------------------
# 7. Save exact Hamiltonian and system diagnostics
# ------------------------------------------------------------

# Recover/rebuild system if needed.
try:
    system_obj = sys_nmr
except NameError:
    system_obj = NMRSystem(CFG, device)

np.save(
    EXPORT_ROOT / "H0_full_XX_YY_ZZ.npy",
    system_obj.H0.detach().cpu().numpy()
)

# Save real/imaginary Hamiltonian tables as CSV too.
H0_np = system_obj.H0.detach().cpu().numpy()

pd.DataFrame(H0_np.real).to_csv(
    TABLE_DIR / "H0_real.csv",
    index=False
)

pd.DataFrame(H0_np.imag).to_csv(
    TABLE_DIR / "H0_imag.csv",
    index=False
)

try:
    diagnostics = system_obj.diagnostics()
except Exception:
    diagnostics = {}

with open(EXPORT_ROOT / "system_diagnostics.json", "w") as f:
    json.dump(
        {k: float(v) if np.isscalar(v) else str(v)
         for k, v in diagnostics.items()},
        f,
        indent=2
    )


# ------------------------------------------------------------
# 8. Save validation table
# ------------------------------------------------------------

validation_df = pd.DataFrame({
    "gate_index": np.arange(len(fids)),
    "q0": q_val_np[:, 0],
    "qx": q_val_np[:, 1],
    "qy": q_val_np[:, 2],
    "qz": q_val_np[:, 3],
    "fidelity": fids,
    "infidelity": 1.0 - fids,
})

validation_df.to_csv(
    TABLE_DIR / "validation_gates_and_fidelities.csv",
    index=False
)

history_df.to_csv(
    TABLE_DIR / "training_validation_history.csv",
    index=False
)


# ------------------------------------------------------------
# 9. Export representative WORST / MEDIAN / BEST gates
# ------------------------------------------------------------

idx_worst = int(np.argmin(fids))
idx_best = int(np.argmax(fids))

median_value = float(np.median(fids))
idx_median = int(
    np.argmin(np.abs(fids - median_value))
)

representatives = {
    "worst": idx_worst,
    "median": idx_median,
    "best": idx_best,
}

representative_summary = {}

for label, idx in representatives.items():

    q_np = q_val_np[idx]
    q = torch.tensor(
        q_np[None, :],
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        controls = model(q)

        target = lift_target(
            quaternion_to_u2(q, system_obj),
            system_obj
        )

        achieved = propagate(
            controls,
            system_obj
        )

        fid_check = unitary_fidelity(
            target,
            achieved
        ).item()

    ctrl = controls[0].detach().cpu().numpy()

    ux = ctrl[:, 0]
    uy = ctrl[:, 1]

    amplitude = np.sqrt(
        ux**2 + uy**2
    )

    phase = np.arctan2(
        uy,
        ux
    )

    phase_unwrapped = np.unwrap(phase)

    time_us = (
        np.arange(len(ux))
        * CFG.dt_s
        * 1e6
    )

    gate_dir = PULSE_DIR / label
    gate_dir.mkdir(exist_ok=True)

    # Raw compact archive.
    np.savez(
        gate_dir / f"{label}_pulse.npz",
        gate_index=idx,
        quaternion=q_np,
        fidelity=fid_check,
        ux_hz=ux,
        uy_hz=uy,
        amplitude_hz=amplitude,
        phase_rad=phase,
        phase_unwrapped_rad=phase_unwrapped,
        time_us=time_us,
        dt_s=CFG.dt_s,
    )

    # Easy-to-read CSV.
    pulse_df = pd.DataFrame({
        "slice": np.arange(len(ux)),
        "time_us": time_us,
        "ux_hz": ux,
        "uy_hz": uy,
        "amplitude_hz": amplitude,
        "phase_rad": phase,
        "phase_unwrapped_rad": phase_unwrapped,
    })

    pulse_df.to_csv(
        gate_dir / f"{label}_pulse.csv",
        index=False
    )

    # Exact target and realized unitaries.
    np.save(
        gate_dir / f"{label}_target_unitary.npy",
        target[0].detach().cpu().numpy()
    )

    np.save(
        gate_dir / f"{label}_achieved_unitary.npy",
        achieved[0].detach().cpu().numpy()
    )

    representative_summary[label] = {
        "gate_index": idx,
        "stored_validation_fidelity": float(fids[idx]),
        "recomputed_fidelity": float(fid_check),
        "quaternion": q_np.tolist(),
        "peak_amplitude_hz": float(amplitude.max()),
        "rms_amplitude_hz": float(
            np.sqrt(np.mean(amplitude**2))
        ),
    }

with open(
    PULSE_DIR / "representative_gate_summary.json",
    "w"
) as f:
    json.dump(
        representative_summary,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 10. Paper-quality figure helper
# ------------------------------------------------------------

def save_current_figure(name):
    plt.tight_layout()

    plt.savefig(
        FIG_DIR / f"{name}.png",
        dpi=600,
        bbox_inches="tight"
    )

    # Vector format for LaTeX / journal production.
    plt.savefig(
        FIG_DIR / f"{name}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close()


# ------------------------------------------------------------
# 11. FIGURE: Training evolution
# ------------------------------------------------------------

if len(history_df) > 0:

    plt.figure(figsize=(7.2, 4.8))

    for col in ["mean", "median", "p05", "p01"]:
        if col in history_df.columns:
            plt.plot(
                history_df["step"],
                history_df[col],
                label=col
            )

    plt.xlabel("Training step")
    plt.ylabel("Held-out unitary fidelity")
    plt.ylim(0.0, 1.005)
    plt.legend()
    plt.grid(alpha=0.2)

    save_current_figure(
        "training_fidelity_history"
    )


# ------------------------------------------------------------
# 12. FIGURE: Tail / minimum fidelity evolution
# ------------------------------------------------------------

if len(history_df) > 0:

    plt.figure(figsize=(7.2, 4.8))

    for col in ["p05", "p01", "min"]:
        if col in history_df.columns:
            plt.plot(
                history_df["step"],
                history_df[col],
                label=col
            )

    plt.xlabel("Training step")
    plt.ylabel("Held-out unitary fidelity")
    plt.ylim(0.0, 1.005)
    plt.legend()
    plt.grid(alpha=0.2)

    save_current_figure(
        "training_tail_fidelity_history"
    )


# ------------------------------------------------------------
# 13. FIGURE: Validation fidelity histogram
# ------------------------------------------------------------

plt.figure(figsize=(7.0, 4.8))

plt.hist(
    fids,
    bins=40
)

plt.axvline(
    summary["mean"],
    linestyle="--",
    label=f"Mean = {summary['mean']:.5f}"
)

plt.axvline(
    summary["p05"],
    linestyle=":",
    label=f"5th percentile = {summary['p05']:.5f}"
)

plt.xlabel("Unitary fidelity")
plt.ylabel("Number of held-out Haar gates")
plt.legend()
plt.grid(alpha=0.15)

save_current_figure(
    "validation_fidelity_histogram"
)


# ------------------------------------------------------------
# 14. FIGURE: Ordered fidelity distribution
# ------------------------------------------------------------

sorted_fids = np.sort(fids)

percentile = (
    np.arange(1, len(sorted_fids) + 1)
    / len(sorted_fids)
    * 100.0
)

plt.figure(figsize=(7.0, 4.8))

plt.plot(
    percentile,
    sorted_fids
)

plt.xlabel("Validation-set percentile (%)")
plt.ylabel("Unitary fidelity")
plt.ylim(
    max(0.0, sorted_fids.min() - 0.02),
    1.001
)
plt.grid(alpha=0.2)

save_current_figure(
    "ordered_validation_fidelity"
)


# ------------------------------------------------------------
# 15. FIGURE: Infidelity distribution on log scale
# ------------------------------------------------------------

infidelity = np.maximum(
    1.0 - fids,
    1e-12
)

sorted_infidelity = np.sort(infidelity)

plt.figure(figsize=(7.0, 4.8))

plt.semilogy(
    percentile,
    sorted_infidelity
)

plt.xlabel("Validation-set percentile (%)")
plt.ylabel("Infidelity (1 - F)")
plt.grid(alpha=0.2)

save_current_figure(
    "ordered_validation_infidelity_log"
)


# ------------------------------------------------------------
# 16. FIGURES: Representative pulses
# ------------------------------------------------------------

for label, idx in representatives.items():

    pulse_npz = np.load(
        PULSE_DIR / label / f"{label}_pulse.npz"
    )

    time_us = pulse_npz["time_us"]
    ux = pulse_npz["ux_hz"]
    uy = pulse_npz["uy_hz"]
    amp = pulse_npz["amplitude_hz"]
    phase = pulse_npz["phase_unwrapped_rad"]

    fid = float(
        pulse_npz["fidelity"]
    )

    # Amplitude
    plt.figure(figsize=(8.0, 4.3))
    plt.plot(time_us, amp)
    plt.xlabel("Time (µs)")
    plt.ylabel("RF amplitude (Hz-equivalent)")
    plt.title(
        f"{label.capitalize()} validation gate, "
        f"F = {fid:.6f}"
    )
    plt.grid(alpha=0.2)

    save_current_figure(
        f"{label}_pulse_amplitude"
    )

    # Phase
    plt.figure(figsize=(8.0, 4.3))
    plt.plot(time_us, phase)
    plt.xlabel("Time (µs)")
    plt.ylabel("Unwrapped RF phase (rad)")
    plt.title(
        f"{label.capitalize()} validation gate, "
        f"F = {fid:.6f}"
    )
    plt.grid(alpha=0.2)

    save_current_figure(
        f"{label}_pulse_phase"
    )

    # I/Q quadratures
    plt.figure(figsize=(8.0, 4.3))
    plt.plot(
        time_us,
        ux,
        label=r"$u_x$"
    )
    plt.plot(
        time_us,
        uy,
        label=r"$u_y$"
    )
    plt.xlabel("Time (µs)")
    plt.ylabel("RF quadrature (Hz-equivalent)")
    plt.title(
        f"{label.capitalize()} validation gate, "
        f"F = {fid:.6f}"
    )
    plt.legend()
    plt.grid(alpha=0.2)

    save_current_figure(
        f"{label}_pulse_IQ"
    )


# ------------------------------------------------------------
# 17. Save software / hardware environment
# ------------------------------------------------------------

environment = {
    "export_timestamp": datetime.now().isoformat(),
    "python": pysys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda_available": torch.cuda.is_available(),
}

if torch.cuda.is_available():
    environment.update({
        "gpu": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "gpu_total_memory_bytes":
            torch.cuda.get_device_properties(0).total_memory,
    })

with open(
    ENV_DIR / "environment.json",
    "w"
) as f:
    json.dump(
        environment,
        f,
        indent=2,
        default=str
    )

try:
    gpu_info = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv"
        ],
        text=True
    )

    with open(
        ENV_DIR / "nvidia_smi.txt",
        "w"
    ) as f:
        f.write(gpu_info)

except Exception as exc:
    with open(
        ENV_DIR / "nvidia_smi_error.txt",
        "w"
    ) as f:
        f.write(str(exc))


try:
    freeze = subprocess.check_output(
        [
            pysys.executable,
            "-m",
            "pip",
            "freeze"
        ],
        text=True
    )

    with open(
        ENV_DIR / "pip_freeze.txt",
        "w"
    ) as f:
        f.write(freeze)

except Exception as exc:
    with open(
        ENV_DIR / "pip_freeze_error.txt",
        "w"
    ) as f:
        f.write(str(exc))


# ------------------------------------------------------------
# 18. Human-readable paper summary
# ------------------------------------------------------------

summary_text = f"""
FULL-HAAR NOMINAL FIDELITY-INFORMED COMPILER
============================================

Best checkpoint step:
{best_ckpt.get("step", "unknown")}

Validation set:
{len(fids)} unseen Haar-random single-qubit gates

FINAL VALIDATION
----------------
Mean fidelity:             {summary['mean']:.9f}
Median fidelity:           {summary['median']:.9f}
Standard deviation:        {summary['std']:.9f}
5th percentile fidelity:   {summary['p05']:.9f}
1st percentile fidelity:   {summary['p01']:.9f}
Minimum fidelity:          {summary['min']:.9f}
Maximum fidelity:          {summary['max']:.9f}

Mean infidelity:           {summary['mean_infidelity']:.9e}
Median infidelity:         {summary['median_infidelity']:.9e}

Fraction F >= 0.95:        {summary['fraction_F_ge_0.950']:.6f}
Fraction F >= 0.97:        {summary['fraction_F_ge_0.970']:.6f}
Fraction F >= 0.98:        {summary['fraction_F_ge_0.980']:.6f}
Fraction F >= 0.99:        {summary['fraction_F_ge_0.990']:.6f}

MODEL
-----
Total parameters:          {n_params:,}
Trainable parameters:      {n_trainable:,}
TorchScript saved:         {traced_saved}

CONTROL MODEL
-------------
Slices:                    {CFG.n_slices}
Slice duration:            {CFG.dt_s:.9e} s
Total pulse duration:      {CFG.n_slices * CFG.dt_s:.9e} s

Hamiltonian:
H0 = pi * sum_i v_i Z_i
   + pi * sum_(i<j) J_ij (X_i X_j + Y_i Y_j + Z_i Z_j)

Target:
U_target = U_Haar x I x I

Control outputs:
u_x(t), u_y(t)

A(t)   = sqrt(u_x(t)^2 + u_y(t)^2)
phi(t) = atan2(u_y(t), u_x(t))

Training supervision:
Differentiable unitary fidelity only.
No GRAPE pulse labels.
No robustness/noise model in this run.
"""

with open(
    EXPORT_ROOT / "PAPER_RUN_SUMMARY.txt",
    "w"
) as f:
    f.write(summary_text.strip() + "\n")


# ------------------------------------------------------------
# 19. Create manifest
# ------------------------------------------------------------

manifest_lines = [
    "FULL-HAAR NOMINAL COMPILER EXPORT",
    "=================================",
    "",
    "Key files:",
    "",
    "model/best_full_checkpoint.pt",
    "  Best model checkpoint including weights, optimizer state,",
    "  configuration, step, and validation statistics.",
    "",
    "model/best_model_state_dict.pt",
    "  Standalone best neural-network weights.",
    "",
    "raw_training_artifacts/history.npy",
    "tables/training_validation_history.csv",
    "  Full stored validation history across training.",
    "",
    "raw_training_artifacts/validation_fidelities.npy",
    "tables/validation_gates_and_fidelities.csv",
    "  Held-out Haar gate results.",
    "",
    "H0_full_XX_YY_ZZ.npy",
    "  Exact 8x8 drift Hamiltonian used in the run.",
    "",
    "representative_pulses/",
    "  Worst, median, and best validation gates with pulse I/Q,",
    "  amplitude, phase, target unitary, and achieved unitary.",
    "",
    "figures/",
    "  600-dpi PNG and vector PDF figures for the paper.",
    "",
    "environment/",
    "  Python/PyTorch/CUDA/GPU and pip package information.",
    "",
    "config.json",
    "  Full experiment configuration.",
    "",
    "final_validation_summary.json",
    "  Machine-readable final statistics.",
    "",
    "PAPER_RUN_SUMMARY.txt",
    "  Human-readable summary of the run.",
]

with open(
    EXPORT_ROOT / "MANIFEST.txt",
    "w"
) as f:
    f.write("\n".join(manifest_lines))


# ------------------------------------------------------------
# 20. Optional SHA256 checksums for archival integrity
# ------------------------------------------------------------

import hashlib

checksum_lines = []

for path in sorted(EXPORT_ROOT.rglob("*")):

    if not path.is_file():
        continue

    h = hashlib.sha256()

    with open(path, "rb") as fh:
        for chunk in iter(
            lambda: fh.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    rel = path.relative_to(EXPORT_ROOT)

    checksum_lines.append(
        f"{h.hexdigest()}  {rel}"
    )

with open(
    EXPORT_ROOT / "SHA256SUMS.txt",
    "w"
) as f:
    f.write(
        "\n".join(checksum_lines)
    )


# ------------------------------------------------------------
# 21. ZIP EVERYTHING
# ------------------------------------------------------------

zip_base = str(EXPORT_ROOT)

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=EXPORT_ROOT.parent,
    base_dir=EXPORT_ROOT.name
)

zip_size_mb = (
    os.path.getsize(zip_path)
    / 1024**2
)

print("\n" + "=" * 70)
print("EXPORT COMPLETE")
print("=" * 70)

print("\nFinal fidelity statistics:")
for k in [
    "mean",
    "median",
    "p05",
    "p01",
    "min",
    "max"
]:
    print(f"  {k:>8s}: {summary[k]:.9f}")

print(f"\nZIP: {zip_path}")
print(f"Size: {zip_size_mb:.2f} MB")
print(f"Files exported: {sum(1 for p in EXPORT_ROOT.rglob('*') if p.is_file())}")

print("\nStarting browser download...")

# Automatically download the complete archive.
files.download(zip_path)